# Diffusion Forcing

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["font.size"] = 16

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

We consider a simple example with 1D time series.

In [ ]:
def create_ts_data(num_samples=1_000):
    x = torch.rand((num_samples)) * torch.pi - torch.pi / 2
    y = torch.sin(x * 2)
    return torch.stack([x, y], dim=1)

In [ ]:
train_dataset = create_ts_data()
train_loader = DataLoader(dataset=train_dataset, batch_size=1024, shuffle=True)

In [ ]:
plt.scatter(train_dataset[:1000, 0], train_dataset[:1000, 1], s=50, alpha=0.5, color="green")
plt.title("Time Series")
plt.grid(alpha=0.1)
plt.gca().set_aspect("equal")
plt.tight_layout()
plt.show() 

Building the model class

In [ ]:
class DiffusionForcingModel(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, num_layers=1):
        super().__init__()
        
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.t_emb = nn.Linear(1, hidden_dim)
        self.act1 = nn.GELU()
        
        self.middle_layer1 = nn.Linear(hidden_dim, hidden_dim)

        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=num_layers, bias=True, batch_first=True)
        self.act2 = nn.GELU()
        
        self.output_layer = nn.Linear(hidden_dim, input_dim)
    
    def forward(self, x, t):
        t_emb = self.t_emb(t)
        x = self.input_layer(x) + t_emb
        x = self.act1(x)

        x = self.middle_layer1(x) + t_emb
        x, (h, c) = self.lstm(x)
        x = x + t_emb
        x = self.act2(x)
        
        x = self.output_layer(x)
        return x

Training

In [ ]:
epochs = 10_000
learning_rate = 3e-4

model = DiffusionForcingModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

model.train()

for epoch in tqdm(range(epochs), desc="Epoch"):
    for x_1 in train_loader:
        x_1 = x_1.to(device)
        x_0 = torch.randn_like(x_1).to(device)
        t = torch.rand(x_1.shape[0], 1).to(device)
        x_t = t * x_1 + (1 - t) * x_0
        velocity = x_1 - x_0
        pred_velocity = model(x_t, t)
        loss = F.mse_loss(pred_velocity, velocity)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Sample from trained model

In [ ]:
num_samples = 1_000
xT = torch.randn((num_samples, 2)).to(device)
xT[:, 0] = torch.sort(xT[:, 0])[0]

plt.scatter(xT[:, 0].cpu().flatten(), xT[:, 1].cpu().flatten())
plt.title("Initial Distribution")
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

In [ ]:
x_ts = [torch.clone(xT)]

t = torch.zeros((num_samples, 1))

window_size = 64
h = 1 / window_size

model.eval()
for i in tqdm(range(1, num_samples + window_size)):
    left = max(0, i - window_size)
    right = min(num_samples, i)
    
    xT_window = xT[left:right]
    t_window = t[left:right].to(device)

    with torch.no_grad():
        pred = model(xT_window, t_window)
    
    xT[left:right] = xT_window + h * pred 
    t[left:right] = t_window + h

    if i % (num_samples / 100) == 0:
        x_ts.append(torch.clone(xT))

In [ ]:
plt.scatter(xT[:, 0].cpu().flatten(), xT[:, 1].cpu().flatten())
plt.title("Forecasting")
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(5, 5))

x_gen = torch.clone(xT)
plt.scatter(x_gen[:, 0].cpu(), x_gen[:, 1].cpu(), s=1)
plt.scatter(x_ts[0][:, 0].cpu(), x_ts[0][:, 1].cpu(), s=1)

step = 1

for sample_i in tqdm(range(num_samples)):
    for t in range(0, len(x_ts) - step - 1, step):
        plt.plot(
            [x_ts[t][sample_i, 0].item(), x_ts[t + step][sample_i, 0].item()], 
            [x_ts[t][sample_i, 1].item(), x_ts[t + step][sample_i, 1].item()], 
            c="Green",
            alpha=0.1
        )

plt.title("Learned Trajectories")
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

In [ ]:
len(x_ts)

In [ ]:
for i in range(len(x_ts)):
    if i % 10 == 0 or i == len(x_ts):
        plt.scatter(x_ts[i][:, 0].cpu(), x_ts[i][:, 1].cpu(), s=1)
        plt.title(f"Forecasting, step {i}")
        plt.grid(alpha=0.1)
        plt.tight_layout()
        plt.show() 